# 02 — Ingest Subscription Events to Bronze

## Purpose

This notebook incrementally ingests raw subscription snapshots and CDC events from the subscription-management system into an append-only Bronze Delta table using Databricks Auto Loader.

The Bronze layer preserves contractual, pricing, billing, lifecycle, and CDC values exactly as received. Technical metadata is added for file lineage, ingestion auditing, schema drift detection, duplicate analysis, and downstream subscription-state processing.

## Business Grain

One row represents one raw subscription source event.

A subscription can appear multiple times because Bronze preserves its initial INSERT event and every subsequent INSERT or UPDATE event, including plan changes, cancellations, pauses, and newly created subscriptions.

## Ingestion Design

- Explicit subscription event schema
- Incremental JSON ingestion with Auto Loader
- Checkpoint-based exactly-once file processing
- `AvailableNow` trigger for Serverless compatibility
- Rescued-data capture for unexpected fields
- Source-file and ingestion metadata
- Deterministic record hashes
- Source-to-Bronze reconciliation
- Idempotent rerun validation

## Source

- `/Volumes/workspace/revenue_leakage_bronze/landing/subscription_system/subscriptions`

## Target

- `workspace.revenue_leakage_bronze.subscription_events`

## Expected First-Load Volume

- 6,000 initial subscription events
- 550 CDC events
- 6,550 total Bronze events

## 1. Configuration and Explicit Source Schema

Define the subscription source, Auto Loader state locations, Bronze target, expected volumes, and complete subscription event data contract.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    TimestampType,
    IntegerType,
    DecimalType,
    BooleanType,
)

SOURCE_SYSTEM = "Subscription System"
SOURCE_ENTITY = "subscriptions"
SOURCE_FORMAT = "json"

EXPECTED_INITIAL_SUBSCRIPTION_COUNT = 6000
EXPECTED_SUBSCRIPTION_CDC_COUNT = 550
EXPECTED_TOTAL_SUBSCRIPTION_EVENT_COUNT = 6550

EXPECTED_OPERATION_COUNTS = {
    "INSERT": 6250,
    "UPDATE": 300,
}

LANDING_PATH = (
    "/Volumes/workspace/"
    "revenue_leakage_bronze/landing"
)

SUBSCRIPTIONS_SOURCE_PATH = (
    f"{LANDING_PATH}/"
    "subscription_system/subscriptions"
)

SUBSCRIPTIONS_INITIAL_PATH = (
    f"{SUBSCRIPTIONS_SOURCE_PATH}/initial_load"
)

SUBSCRIPTIONS_CHANGE_BATCH_PATH = (
    f"{SUBSCRIPTIONS_SOURCE_PATH}/change_batch_001"
)

SUBSCRIPTIONS_SCHEMA_PATH = (
    f"{LANDING_PATH}/_schemas/"
    "bronze/subscription_system/subscriptions"
)

SUBSCRIPTIONS_CHECKPOINT_PATH = (
    f"{LANDING_PATH}/_checkpoints/"
    "bronze/subscription_system/subscriptions"
)

SUBSCRIPTIONS_BRONZE_TABLE = (
    "workspace.revenue_leakage_bronze."
    "subscription_events"
)

SUBSCRIPTION_EVENT_SCHEMA = StructType([
    StructField(
        "subscription_id",
        StringType(),
        False
    ),
    StructField(
        "customer_id",
        StringType(),
        False
    ),
    StructField(
        "plan_id",
        StringType(),
        False
    ),
    StructField(
        "plan_name",
        StringType(),
        False
    ),
    StructField(
        "start_date",
        DateType(),
        False
    ),
    StructField(
        "end_date",
        DateType(),
        True
    ),
    StructField(
        "subscription_status",
        StringType(),
        False
    ),
    StructField(
        "billing_frequency",
        StringType(),
        False
    ),
    StructField(
        "billing_day",
        IntegerType(),
        False
    ),
    StructField(
        "base_monthly_price",
        DecimalType(10, 2),
        False
    ),
    StructField(
        "discount_percentage",
        DecimalType(5, 2),
        False
    ),
    StructField(
        "contracted_monthly_price",
        DecimalType(10, 2),
        False
    ),
    StructField(
        "contracted_billing_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "discount_start_date",
        DateType(),
        True
    ),
    StructField(
        "discount_end_date",
        DateType(),
        True
    ),
    StructField(
        "included_usage_units",
        IntegerType(),
        False
    ),
    StructField(
        "overage_unit_price",
        DecimalType(10, 4),
        False
    ),
    StructField(
        "payment_terms_days",
        IntegerType(),
        False
    ),
    StructField(
        "auto_renew",
        BooleanType(),
        False
    ),
    StructField(
        "currency",
        StringType(),
        False
    ),
    StructField(
        "operation",
        StringType(),
        False
    ),
    StructField(
        "event_timestamp",
        TimestampType(),
        False
    ),
    StructField(
        "snapshot_date",
        DateType(),
        False
    ),
])

SUBSCRIPTION_SOURCE_COLUMNS = (
    SUBSCRIPTION_EVENT_SCHEMA.fieldNames()
)

BRONZE_METADATA_COLUMNS = [
    "_source_system",
    "_source_entity",
    "_source_file_path",
    "_source_file_name",
    "_source_file_size",
    "_source_file_modification_time",
    "_ingested_at",
    "_ingestion_date",
    "_record_hash",
    "_rescued_data",
]

SUBSCRIPTION_BRONZE_COLUMNS = (
    SUBSCRIPTION_SOURCE_COLUMNS
    + BRONZE_METADATA_COLUMNS
)

## 2. Load and Validate the Subscription Source

Load the initial subscription snapshot and CDC batch using the explicit schema, then validate expected volumes, identifiers, required fields, domain values, dates, pricing, and event uniqueness before Bronze ingestion.

In [0]:
subscriptions_initial_source_df = (
    spark.read
    .format(SOURCE_FORMAT)
    .schema(SUBSCRIPTION_EVENT_SCHEMA)
    .load(SUBSCRIPTIONS_INITIAL_PATH)
    .withColumn("_landing_batch", F.lit("initial_load"))
)

subscriptions_changes_source_df = (
    spark.read
    .format(SOURCE_FORMAT)
    .schema(SUBSCRIPTION_EVENT_SCHEMA)
    .load(SUBSCRIPTIONS_CHANGE_BATCH_PATH)
    .withColumn("_landing_batch", F.lit("change_batch_001"))
)

subscriptions_source_df = (
    subscriptions_initial_source_df
    .unionByName(subscriptions_changes_source_df)
)

nullable_source_columns = {
    "end_date",
    "discount_start_date",
    "discount_end_date",
}

required_source_columns = [
    column_name
    for column_name in SUBSCRIPTION_SOURCE_COLUMNS
    if column_name not in nullable_source_columns
]

required_field_is_missing = None

for column_name in required_source_columns:
    missing_condition = (
        F.col(column_name).isNull()
        | (
            F.trim(F.col(column_name).cast("string"))
            == F.lit("")
        )
    )

    required_field_is_missing = (
        missing_condition
        if required_field_is_missing is None
        else required_field_is_missing | missing_condition
    )

valid_plan_mapping_condition = (
    (
        (F.col("plan_id") == "PLAN_STARTER")
        & (F.col("plan_name") == "Starter")
    )
    | (
        (F.col("plan_id") == "PLAN_GROWTH")
        & (F.col("plan_name") == "Growth")
    )
    | (
        (F.col("plan_id") == "PLAN_PROFESSIONAL")
        & (F.col("plan_name") == "Professional")
    )
    | (
        (F.col("plan_id") == "PLAN_ENTERPRISE")
        & (F.col("plan_name") == "Enterprise")
    )
)

invalid_domain_condition = (
    (~valid_plan_mapping_condition)
    | (
        ~F.col("subscription_status").isin(
            "Active",
            "Paused",
            "Cancelled",
        )
    )
    | (
        ~F.col("billing_frequency").isin(
            "Monthly",
            "Annual",
        )
    )
    | (
        ~F.col("payment_terms_days").isin(
            15,
            30,
            45,
        )
    )
    | (F.col("currency") != "USD")
    | (
        ~F.col("operation").isin(
            "INSERT",
            "UPDATE",
        )
    )
    | (F.col("billing_day") < 1)
    | (F.col("billing_day") > 28)
)

invalid_pricing_condition = (
    (F.col("base_monthly_price") <= 0)
    | (F.col("discount_percentage") < 0)
    | (F.col("discount_percentage") > 100)
    | (F.col("contracted_monthly_price") < 0)
    | (F.col("contracted_billing_amount") <= 0)
    | (F.col("included_usage_units") < 0)
    | (F.col("overage_unit_price") < 0)
)

invalid_date_condition = (
    (F.col("start_date") > F.col("snapshot_date"))
    | (
        F.col("end_date").isNotNull()
        & (F.col("end_date") < F.col("start_date"))
    )
    | (
        (F.col("subscription_status") == "Cancelled")
        & F.col("end_date").isNull()
    )
    | (
        F.col("discount_start_date").isNotNull()
        & F.col("discount_end_date").isNotNull()
        & (
            F.col("discount_end_date")
            < F.col("discount_start_date")
        )
    )
    | (
        F.to_date("event_timestamp")
        > F.col("snapshot_date")
    )
)

source_metrics = (
    subscriptions_source_df
    .agg(
        F.count("*").alias("total_event_count"),

        F.sum(
            F.when(
                F.col("_landing_batch") == "initial_load",
                1,
            ).otherwise(0)
        ).alias("initial_event_count"),

        F.sum(
            F.when(
                F.col("_landing_batch") == "change_batch_001",
                1,
            ).otherwise(0)
        ).alias("cdc_event_count"),

        F.countDistinct(
            F.struct(
                "subscription_id",
                "operation",
                "event_timestamp",
            )
        ).alias("distinct_event_key_count"),

        F.countDistinct(
            F.when(
                F.col("_landing_batch") == "initial_load",
                F.col("subscription_id"),
            )
        ).alias("distinct_initial_subscription_count"),

        F.sum(
            F.when(
                required_field_is_missing,
                1,
            ).otherwise(0)
        ).alias("null_required_field_count"),

        F.sum(
            F.when(
                ~F.col("subscription_id").rlike(
                    r"^S[0-9]{7}$"
                ),
                1,
            ).otherwise(0)
        ).alias("invalid_subscription_id_count"),

        F.sum(
            F.when(
                ~F.col("customer_id").rlike(
                    r"^C[0-9]{6}$"
                ),
                1,
            ).otherwise(0)
        ).alias("invalid_customer_id_count"),

        F.sum(
            F.when(
                invalid_domain_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_domain_count"),

        F.sum(
            F.when(
                invalid_pricing_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_pricing_count"),

        F.sum(
            F.when(
                invalid_date_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_date_count"),

        F.sum(
            F.when(
                (
                    F.col("_landing_batch")
                    == "initial_load"
                )
                & (F.col("operation") != "INSERT"),
                1,
            ).otherwise(0)
        ).alias("invalid_initial_operation_count"),
    )
    .first()
    .asDict()
)

initial_event_count = int(
    source_metrics["initial_event_count"]
)

cdc_event_count = int(
    source_metrics["cdc_event_count"]
)

total_event_count = int(
    source_metrics["total_event_count"]
)

distinct_event_key_count = int(
    source_metrics["distinct_event_key_count"]
)

duplicate_source_event_count = (
    total_event_count
    - distinct_event_key_count
)

duplicate_initial_subscription_count = (
    initial_event_count
    - int(
        source_metrics[
            "distinct_initial_subscription_count"
        ]
    )
)

operation_counts_df = (
    subscriptions_source_df
    .groupBy("operation")
    .count()
    .orderBy("operation")
)

actual_operation_counts = {
    row["operation"]: int(row["count"])
    for row in operation_counts_df.collect()
}

assert (
    initial_event_count
    == EXPECTED_INITIAL_SUBSCRIPTION_COUNT
), (
    "Unexpected initial subscription count: "
    f"{initial_event_count}"
)

assert (
    cdc_event_count
    == EXPECTED_SUBSCRIPTION_CDC_COUNT
), (
    "Unexpected subscription CDC count: "
    f"{cdc_event_count}"
)

assert (
    total_event_count
    == EXPECTED_TOTAL_SUBSCRIPTION_EVENT_COUNT
), (
    "Unexpected total subscription event count: "
    f"{total_event_count}"
)

assert duplicate_source_event_count == 0, (
    "Duplicate subscription source events found: "
    f"{duplicate_source_event_count}"
)

assert duplicate_initial_subscription_count == 0, (
    "Duplicate initial subscription IDs found: "
    f"{duplicate_initial_subscription_count}"
)

assert actual_operation_counts == EXPECTED_OPERATION_COUNTS, (
    "Unexpected operation counts: "
    f"{actual_operation_counts}"
)

validation_error_columns = [
    "null_required_field_count",
    "invalid_subscription_id_count",
    "invalid_customer_id_count",
    "invalid_domain_count",
    "invalid_pricing_count",
    "invalid_date_count",
    "invalid_initial_operation_count",
]

for error_column in validation_error_columns:
    error_count = int(source_metrics[error_column])

    assert error_count == 0, (
        f"{error_column}: {error_count}"
    )

print(
    f"Initial subscription events: "
    f"{initial_event_count:,}"
)

print(
    f"Subscription CDC events: "
    f"{cdc_event_count:,}"
)

print(
    f"Total landing events: "
    f"{total_event_count:,}"
)

print(
    f"Distinct source event keys: "
    f"{distinct_event_key_count:,}"
)

print(
    "Null required source fields: "
    f"{source_metrics['null_required_field_count']:,}"
)

print(
    "Duplicate source events: "
    f"{duplicate_source_event_count:,}"
)

print(
    "Duplicate initial subscription IDs: "
    f"{duplicate_initial_subscription_count:,}"
)

print(
    "Invalid subscription IDs: "
    f"{source_metrics['invalid_subscription_id_count']:,}"
)

print(
    "Invalid customer IDs: "
    f"{source_metrics['invalid_customer_id_count']:,}"
)

print(
    "Invalid domain values: "
    f"{source_metrics['invalid_domain_count']:,}"
)

print(
    "Invalid pricing values: "
    f"{source_metrics['invalid_pricing_count']:,}"
)

print(
    "Invalid date relationships: "
    f"{source_metrics['invalid_date_count']:,}"
)

display(
    subscriptions_source_df
    .groupBy(
        "_landing_batch",
        "operation",
    )
    .count()
    .orderBy(
        "_landing_batch",
        "operation",
    )
)

## 3. Ingest Subscription Events into Bronze

Use Databricks Auto Loader to incrementally ingest the subscription JSON files into a Delta Bronze table. Add technical lineage metadata, rescued-data support, and a deterministic record hash while maintaining idempotency through a dedicated checkpoint.

In [0]:
record_hash_columns = [
    F.coalesce(
        F.col(column_name).cast("string"),
        F.lit("<NULL>"),
    )
    for column_name in SUBSCRIPTION_SOURCE_COLUMNS
]

subscription_events_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option(
        "cloudFiles.format",
        SOURCE_FORMAT,
    )
    .option(
        "cloudFiles.schemaLocation",
        SUBSCRIPTIONS_SCHEMA_PATH,
    )
    .option(
        "cloudFiles.schemaEvolutionMode",
        "rescue",
    )
    .option(
        "rescuedDataColumn",
        "_rescued_data",
    )
    .schema(SUBSCRIPTION_EVENT_SCHEMA)
    .load(SUBSCRIPTIONS_SOURCE_PATH)
    .withColumn(
        "_source_system",
        F.lit(SOURCE_SYSTEM),
    )
    .withColumn(
        "_source_entity",
        F.lit(SOURCE_ENTITY),
    )
    .withColumn(
        "_source_file_path",
        F.col("_metadata.file_path"),
    )
    .withColumn(
        "_source_file_name",
        F.col("_metadata.file_name"),
    )
    .withColumn(
        "_source_file_size",
        F.col("_metadata.file_size"),
    )
    .withColumn(
        "_source_file_modification_time",
        F.col("_metadata.file_modification_time"),
    )
    .withColumn(
        "_ingested_at",
        F.current_timestamp(),
    )
    .withColumn(
        "_ingestion_date",
        F.to_date("_ingested_at"),
    )
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *record_hash_columns,
            ),
            256,
        ),
    )
    .select(
        *SUBSCRIPTION_BRONZE_COLUMNS
    )
)

subscription_bronze_query = (
    subscription_events_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        SUBSCRIPTIONS_CHECKPOINT_PATH,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        SUBSCRIPTIONS_BRONZE_TABLE
    )
)

subscription_bronze_query.awaitTermination()

print(
    "Subscription Auto Loader ingestion completed."
)

print(
    f"Bronze table: "
    f"{SUBSCRIPTIONS_BRONZE_TABLE}"
)

print(
    f"Canonical source: "
    f"{SUBSCRIPTIONS_SOURCE_PATH}"
)

print(
    f"Checkpoint: "
    f"{SUBSCRIPTIONS_CHECKPOINT_PATH}"
)

## 4. Validate and Reconcile the Bronze Subscription Table

Validate Bronze row counts, event uniqueness, required source and metadata fields, rescued data, canonical lineage, operation distribution, Delta format, and exact source-to-Bronze content reconciliation.

In [0]:
bronze_subscriptions_df = spark.table(
    SUBSCRIPTIONS_BRONZE_TABLE
)

required_metadata_columns = [
    "_source_system",
    "_source_entity",
    "_source_file_path",
    "_source_file_name",
    "_source_file_size",
    "_source_file_modification_time",
    "_ingested_at",
    "_ingestion_date",
    "_record_hash",
]

required_metadata_is_missing = None

for column_name in required_metadata_columns:
    missing_condition = (
        F.col(column_name).isNull()
        | (
            F.trim(F.col(column_name).cast("string"))
            == F.lit("")
        )
    )

    required_metadata_is_missing = (
        missing_condition
        if required_metadata_is_missing is None
        else required_metadata_is_missing | missing_condition
    )

invalid_lineage_condition = (
    (F.col("_source_system") != SOURCE_SYSTEM)
    | (F.col("_source_entity") != SOURCE_ENTITY)
    | (
        ~F.col("_source_file_path").contains(
            SUBSCRIPTIONS_SOURCE_PATH
        )
    )
)

bronze_metrics = (
    bronze_subscriptions_df
    .agg(
        F.count("*").alias("bronze_event_count"),

        F.countDistinct(
            F.struct(
                "subscription_id",
                "operation",
                "event_timestamp",
            )
        ).alias("distinct_event_key_count"),

        F.countDistinct(
            "_record_hash"
        ).alias("distinct_record_hash_count"),

        F.sum(
            F.when(
                required_field_is_missing,
                1,
            ).otherwise(0)
        ).alias("null_required_source_field_count"),

        F.sum(
            F.when(
                required_metadata_is_missing,
                1,
            ).otherwise(0)
        ).alias("null_required_metadata_field_count"),

        F.sum(
            F.when(
                F.col("_rescued_data").isNotNull()
                & (
                    F.trim(F.col("_rescued_data"))
                    != F.lit("")
                ),
                1,
            ).otherwise(0)
        ).alias("rescued_data_row_count"),

        F.sum(
            F.when(
                invalid_lineage_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_lineage_row_count"),

        F.sum(
            F.when(
                F.col("_source_file_path").contains(
                    "/initial_load/"
                ),
                1,
            ).otherwise(0)
        ).alias("initial_load_event_count"),

        F.sum(
            F.when(
                F.col("_source_file_path").contains(
                    "/change_batch_001/"
                ),
                1,
            ).otherwise(0)
        ).alias("cdc_event_count"),
    )
    .first()
    .asDict()
)

source_reconciliation_df = (
    subscriptions_source_df
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(
                        F.col(column_name).cast("string"),
                        F.lit("<NULL>"),
                    )
                    for column_name
                    in SUBSCRIPTION_SOURCE_COLUMNS
                ],
            ),
            256,
        ),
    )
    .select(
        "subscription_id",
        "operation",
        "event_timestamp",
        "_record_hash",
    )
)

bronze_reconciliation_df = (
    bronze_subscriptions_df
    .select(
        "subscription_id",
        "operation",
        "event_timestamp",
        "_record_hash",
    )
)

source_missing_from_bronze_count = (
    source_reconciliation_df
    .exceptAll(bronze_reconciliation_df)
    .count()
)

unexpected_bronze_record_count = (
    bronze_reconciliation_df
    .exceptAll(source_reconciliation_df)
    .count()
)

source_bronze_mismatch_count = (
    source_missing_from_bronze_count
    + unexpected_bronze_record_count
)

bronze_operation_counts_df = (
    bronze_subscriptions_df
    .groupBy("operation")
    .count()
    .orderBy("operation")
)

actual_bronze_operation_counts = {
    row["operation"]: int(row["count"])
    for row in bronze_operation_counts_df.collect()
}

target_detail = (
    spark.sql(
        f"DESCRIBE DETAIL {SUBSCRIPTIONS_BRONZE_TABLE}"
    )
    .select("format")
    .first()
)

target_format = target_detail["format"].lower()

bronze_event_count = int(
    bronze_metrics["bronze_event_count"]
)

distinct_event_key_count = int(
    bronze_metrics["distinct_event_key_count"]
)

distinct_record_hash_count = int(
    bronze_metrics["distinct_record_hash_count"]
)

assert (
    bronze_event_count
    == EXPECTED_TOTAL_SUBSCRIPTION_EVENT_COUNT
), (
    "Unexpected Bronze subscription count: "
    f"{bronze_event_count}"
)

assert (
    distinct_event_key_count
    == EXPECTED_TOTAL_SUBSCRIPTION_EVENT_COUNT
), (
    "Duplicate Bronze event keys found."
)

assert (
    distinct_record_hash_count
    == EXPECTED_TOTAL_SUBSCRIPTION_EVENT_COUNT
), (
    "Duplicate Bronze record hashes found."
)

assert (
    int(
        bronze_metrics[
            "null_required_source_field_count"
        ]
    )
    == 0
), "Null required source fields found in Bronze."

assert (
    int(
        bronze_metrics[
            "null_required_metadata_field_count"
        ]
    )
    == 0
), "Null required metadata fields found."

assert (
    int(bronze_metrics["rescued_data_row_count"])
    == 0
), "Rescued-data rows found."

assert (
    int(bronze_metrics["invalid_lineage_row_count"])
    == 0
), "Invalid Bronze lineage rows found."

assert (
    int(bronze_metrics["initial_load_event_count"])
    == EXPECTED_INITIAL_SUBSCRIPTION_COUNT
), "Unexpected initial-load Bronze count."

assert (
    int(bronze_metrics["cdc_event_count"])
    == EXPECTED_SUBSCRIPTION_CDC_COUNT
), "Unexpected CDC Bronze count."

assert (
    actual_bronze_operation_counts
    == EXPECTED_OPERATION_COUNTS
), (
    "Unexpected Bronze operation counts: "
    f"{actual_bronze_operation_counts}"
)

assert source_bronze_mismatch_count == 0, (
    "Source/Bronze content mismatches found: "
    f"{source_bronze_mismatch_count}"
)

assert target_format == "delta", (
    f"Unexpected target format: {target_format}"
)

print(
    f"Bronze subscription events: "
    f"{bronze_event_count:,}"
)

print(
    f"Distinct Bronze event keys: "
    f"{distinct_event_key_count:,}"
)

print(
    f"Distinct record hashes: "
    f"{distinct_record_hash_count:,}"
)

print(
    "Null required source fields: "
    f"{bronze_metrics['null_required_source_field_count']:,}"
)

print(
    "Null required metadata fields: "
    f"{bronze_metrics['null_required_metadata_field_count']:,}"
)

print(
    "Rescued-data rows: "
    f"{bronze_metrics['rescued_data_row_count']:,}"
)

print(
    "Invalid canonical lineage rows: "
    f"{bronze_metrics['invalid_lineage_row_count']:,}"
)

print(
    f"Initial-load events: "
    f"{bronze_metrics['initial_load_event_count']:,}"
)

print(
    f"CDC events: "
    f"{bronze_metrics['cdc_event_count']:,}"
)

print(
    f"Source/Bronze content mismatches: "
    f"{source_bronze_mismatch_count:,}"
)

print(
    f"Target format: {target_format}"
)

display(
    bronze_operation_counts_df
)

## 5. Validate Auto Loader Idempotency

Rerun the subscription ingestion using the existing Auto Loader checkpoint and confirm that previously processed source files are not ingested again.

In [0]:
rows_before_idempotency_rerun = (
    spark.table(SUBSCRIPTIONS_BRONZE_TABLE)
    .count()
)

subscription_idempotency_query = (
    subscription_events_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        SUBSCRIPTIONS_CHECKPOINT_PATH,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        SUBSCRIPTIONS_BRONZE_TABLE
    )
)

subscription_idempotency_query.awaitTermination()

rows_after_idempotency_rerun = (
    spark.table(SUBSCRIPTIONS_BRONZE_TABLE)
    .count()
)

rows_added_during_rerun = (
    rows_after_idempotency_rerun
    - rows_before_idempotency_rerun
)

assert (
    rows_before_idempotency_rerun
    == EXPECTED_TOTAL_SUBSCRIPTION_EVENT_COUNT
), (
    "Unexpected row count before rerun: "
    f"{rows_before_idempotency_rerun}"
)

assert (
    rows_after_idempotency_rerun
    == EXPECTED_TOTAL_SUBSCRIPTION_EVENT_COUNT
), (
    "Unexpected row count after rerun: "
    f"{rows_after_idempotency_rerun}"
)

assert rows_added_during_rerun == 0, (
    "Auto Loader idempotency failed. "
    f"Rows added during rerun: "
    f"{rows_added_during_rerun}"
)

print(
    "Rows before idempotency rerun: "
    f"{rows_before_idempotency_rerun:,}"
)

print(
    "Rows after idempotency rerun: "
    f"{rows_after_idempotency_rerun:,}"
)

print(
    "Rows added during rerun: "
    f"{rows_added_during_rerun:,}"
)

print(
    "Subscription Bronze ingestion is idempotent."
)